# Ensaio 3 · Um motor não é outro motor

**Pergunta:** um classificador treinado no rolamento do lado acionado (SKF 6205, sensor DE) continua acertando quando muda a carga, ou quando o rolamento é outro (SKF 6203, lado do ventilador, sensor FE)?

**Dois modelos, mesmo algoritmo (floresta aleatória):**
- **Ingênuo:** recebe o espectro de envelope em Hz fixos (0–500 Hz, colunas de 5 Hz).
- **Com física:** recebe só 4 números, a altura do pico em BPFO, 2×BPFO, BPFI e 2×BPFI, calculados com a geometria e a rotação **do rolamento medido** (multiplicadores oficiais da CWRU).

**Classes:** sem defeito, pista interna, pista externa. A esfera fica de fora porque o ensaio 2 mostrou que nem o diagnóstico clássico a encontra com segurança.

**Regra contra vazamento:** treino e teste separados por arquivo. A única exceção é o controle C, feito de propósito para mostrar o "99%" dos tutoriais.

In [1]:
import sys, json
sys.path.insert(0, "..")
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from src.envelope import *

MULTS = {"DE": MULT, "FE": MULT_FE}
WIN, NWIN = 12_000, 10
DE_FILES = {97: 0, 100: 0, 105: 1, 108: 1, 209: 1, 212: 1, 130: 2, 133: 2, 234: 2, 237: 2}
FE_FILES = {97: 0, 100: 0, 278: 1, 281: 1, 270: 1, 273: 1, 294: 2, 297: 2, 315: 2}
LOAD3 = {100, 108, 212, 133, 237}

In [2]:
def feats(x, rpm, ch):
    f, S = envelope_spectrum(x)
    band = (f > 5) & (f < 500); med = np.median(S[band]) + 1e-12
    edges = np.arange(0, 505, 5)
    naive = [np.log(S[(f >= edges[i]) & (f < edges[i+1])].max() / med + 1e-9) for i in range(len(edges) - 1)]
    fr = rpm / 60; phys = []
    for k in ["BPFO", "BPFI"]:
        for h in [1, 2]:
            ft = MULTS[ch][k] * fr * h; w = np.abs(f - ft) <= 0.02 * ft
            phys.append(np.log(S[w].max() / med))
    return np.array(naive), np.array(phys)

def dataset(files, ch):
    Xn, Xp, y, src, wi = [], [], [], [], []
    for n, lab in files.items():
        x, rpm = load_channel(DATA_DIR / f"{n}.mat", ch)
        for i in range(min(NWIN, len(x) // WIN)):
            a, b = feats(x[i*WIN:(i+1)*WIN], rpm, ch)
            Xn.append(a); Xp.append(b); y.append(lab); src.append(n); wi.append(i)
    return np.array(Xn), np.array(Xp), np.array(y), np.array(src), np.array(wi)

def evaluate(Xtr, ytr, Xte, yte):
    p = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xtr, ytr).predict(Xte)
    fault = yte != 0
    return {"acerto": round(accuracy_score(yte, p), 3),
            "falso_negativo": round(float(np.mean(p[fault] == 0)), 3),
            "falso_positivo": round(float(np.mean(p[~fault] != 0)), 3),
            "matriz": confusion_matrix(yte, p, labels=[0, 1, 2]).tolist()}

In [3]:
Dn, Dp, Dy, Ds, Dw = dataset(DE_FILES, "DE")
Fn, Fp, Fy, _, _ = dataset(FE_FILES, "FE")
tr, te = ~np.isin(Ds, list(LOAD3)), np.isin(Ds, list(LOAD3))
c_tr, c_te = Dw < 5, Dw >= 5
res = {
  "C · mesmos arquivos (controle)": {"ingenuo": evaluate(Dn[c_tr], Dy[c_tr], Dn[c_te], Dy[c_te]), "fisica": evaluate(Dp[c_tr], Dy[c_tr], Dp[c_te], Dy[c_te])},
  "A · mesmo rolamento, carga nova": {"ingenuo": evaluate(Dn[tr], Dy[tr], Dn[te], Dy[te]), "fisica": evaluate(Dp[tr], Dy[tr], Dp[te], Dy[te])},
  "B · rolamento novo (6203, FE)": {"ingenuo": evaluate(Dn, Dy, Fn, Fy), "fisica": evaluate(Dp, Dy, Fp, Fy)},
}
for k, v in res.items():
    print(k)
    for m, r in v.items():
        print(f"   {m:8s} acerto {r['acerto']:.0%} · falso negativo {r['falso_negativo']:.0%} · falso positivo {r['falso_positivo']:.0%}")
json.dump(res, open("../results/ensaio3.json", "w"), indent=1, ensure_ascii=False)

C · mesmos arquivos (controle)
   ingenuo  acerto 100% · falso negativo 0% · falso positivo 0%
   fisica   acerto 100% · falso negativo 0% · falso positivo 0%
A · mesmo rolamento, carga nova
   ingenuo  acerto 26% · falso negativo 60% · falso positivo 0%
   fisica   acerto 100% · falso negativo 0% · falso positivo 0%
B · rolamento novo (6203, FE)
   ingenuo  acerto 33% · falso negativo 83% · falso positivo 0%
   fisica   acerto 100% · falso negativo 0% · falso positivo 0%


## Resultado

| Teste | Ingênuo | Com física |
|---|---|---|
| C · mesmos arquivos (controle) | 100% | 100% |
| A · mesmo rolamento, carga nova | **26%** (60% das falhas viram "normal") | 100% |
| B · rolamento novo | **33%** (83% das falhas viram "normal") | 100% |

**Leitura:** o modelo ingênuo aprendeu "pico em 162 Hz = pista interna". Quando a carga muda, a rotação cai uns 4% e o pico anda alguns Hz. Quando o rolamento muda, a frequência de falha é outra. O modelo continua procurando no lugar antigo e chama a falha de "normal": **falso negativo**, o erro mais caro na manutenção.

O modelo com física recebe a frequência certa para cada rolamento, e aí a mesma floresta generaliza.

**Limites:** 100 janelas no treino e 90 no teste; defeitos artificiais; mesmo motor e mesma bancada (o rolamento e o sensor mudam, o motor não); o modelo com física depende de saber a geometria do rolamento e a rotação. O próximo passo é testar num motor de outra bancada.